In [1]:
import os
import sys

import torch
import numpy as np
import pandas as pd

import shared.utils as su

In [81]:
data_dir = "/scratch/shared/beegfs/piyush/datasets/Wan2.2"
video_dir = f"{data_dir}/videos"
metad_dir = f"{data_dir}/metadata"

In [82]:
!ls $video_dir | wc -l

1030


In [83]:
df_pairs = pd.read_csv("/users/piyush/projects/Wan2.2/data/anchor-hard_negatives-1k.csv")
df_pairs.shape, df_pairs.iloc[0].to_dict()

((982, 2),
 {'sent0': 'The toddler picks a cloth from the chair with his right hand.',
  'hard_neg': 'The toddler places the cloth on the chair with his right hand.'})

In [84]:
# df = pd.read_csv(f"{data_dir}/metadata/video_anchor_tracking.csv")
df = pd.DataFrame(su.io.load_jsonl(f"{data_dir}/metadata/video_anchor_tracking.jsonl"))
df.shape

(519, 7)

In [85]:
video_paths = []
for i in su.log.tqdm_iterator(range(len(df_pairs)), desc='Hunting videos'):
    row = df_pairs.iloc[i].to_dict()
    text_anchor = row['sent0']
    if len(df[df.text_anchor == text_anchor]) == 0:
        # print(f"No video found for row {i}.")
        video_path = None
    else:
        video_path = df[df.text_anchor == text_anchor].iloc[0].video_path
    video_paths.append(video_path)
df_pairs['video0'] = video_paths
n_valid_videos = len(df_pairs) - df_pairs.video0.isnull().sum()
print("Number of valid rows with video found: ", n_valid_videos)

Hunting videos:   0%|          | 0/982 [00:00<?, ?it/s]

Number of valid rows with video found:  514


In [86]:
df_pairs = df_pairs[~df_pairs.video0.isnull()]
df_pairs.shape

(514, 3)

In [87]:
# Visualize a random row

row = df_pairs.sample(n=1).iloc[0].to_dict()
su.visualize.show_forward_reverse(row['video0'], labels=[row['sent0'], row['hard_neg']], width_of_screen=1400)

In [92]:
df_pairs

,sent0,hard_neg,video0
0,The toddler picks a cloth from the chair with ...,The toddler places the cloth on the chair with...,/scratch/shared/beegfs/piyush/datasets/Wan2.2/...
1,The zookeeper places his right hand on the met...,The zookeeper removes his right hand from the ...,/scratch/shared/beegfs/piyush/datasets/Wan2.2/...
2,The laundry worker puts the iron box on the al...,The laundry worker takes the iron box off the ...,/scratch/shared/beegfs/piyush/datasets/Wan2.2/...
3,The plumber opens the tap,The plumber closes the tap,/scratch/shared/beegfs/piyush/datasets/Wan2.2/...
4,The handyman opens a drawer,The handyman closes a drawer,/scratch/shared/beegfs/piyush/datasets/Wan2.2/...
...,...,...,...
918,The student picks a book on the heap of books,The student puts a book back into the heap of ...,/scratch/shared/beegfs/piyush/datasets/Wan2.2/...
919,The seamstress pulls out the needle,The seamstress pushes the needle in,/scratch/shared/beegfs/piyush/datasets/Wan2.2/...
921,The model lifts the hand,The model lowers the hand,/scratch/shared/beegfs/piyush/datasets/Wan2.2/...
922,The cook puts in the machine pan,The cook takes out the machine pan,/scratch/shared/beegfs/piyush/datasets/Wan2.2/...


In [93]:
# Add the reversed video
df_pairs["video_hard_neg"] = df_pairs['video0'].apply(lambda x: x.split(".mp4")[0] + "-reversed.mp4")
df_pairs.rename(columns={"hard_neg": "sent_hard_neg"}, inplace=True)
df_pairs.shape

(514, 4)

In [96]:
df_pairs.video0.apply(os.path.exists).mean(), \
df_pairs.video_hard_neg.apply(os.path.exists).mean()

(1.0, 0.9844357976653697)

In [97]:
df_pairs = df_pairs[df_pairs.video0.apply(os.path.exists)]
df_pairs = df_pairs[df_pairs.video_hard_neg.apply(os.path.exists)]
df_pairs.shape

(506, 4)

In [102]:
i = np.random.randint(len(df_pairs))
row = df_pairs.iloc[i].to_dict()
su.visualize.show_grid_of_videos([row['video0'], row['video_hard_neg']], labels=[row["sent0"], row["sent_hard_neg"]])

In [104]:
row

{'sent0': 'The student drops the cup on the computer desk with his left hand.',
 'sent_hard_neg': 'The student picks up the cup from the computer desk with his left hand.',
 'video0': '/scratch/shared/beegfs/piyush/datasets/Wan2.2/videos/20260504_232838_74477cb6b8aa.mp4',
 'video_hard_neg': '/scratch/shared/beegfs/piyush/datasets/Wan2.2/videos/20260504_232838_74477cb6b8aa-reversed.mp4'}